# Gaze + Head Scene Viewer

交互式查看一个 sequence 某个 frame window 内的：

- 黑色：head/gaze origin trajectory（CPF origin）
- 蓝色：head forward direction
- 红色：gaze direction

使用前提：

1. 已经有 `*_gaze_samples.csv`
2. 已经有 `*_head_samples.csv`
3. notebook 运行在 `adt` 环境里


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from visualization.head_viewer import (
    discover_sequence_ids,
    load_gaze_head_frame,
    plot_gaze_head_scene_window,
    slice_frame_window,
)


In [2]:
REPORTS_DIR = Path('/mnt/d/SparseGaze/ADT-Gaze-structured')
SEQUENCE_IDS = discover_sequence_ids(REPORTS_DIR)
CACHE = {}

def get_frame(sequence_id: str):
    if sequence_id not in CACHE:
        CACHE[sequence_id] = load_gaze_head_frame(REPORTS_DIR, sequence_id)
    return CACHE[sequence_id]

SEQUENCE_IDS[:5], len(SEQUENCE_IDS)


(['Apartment_release_decoration_skeleton_seq131_M1292',
  'Apartment_release_decoration_skeleton_seq132_M1292',
  'Apartment_release_decoration_skeleton_seq133_M1292',
  'Apartment_release_decoration_skeleton_seq134_M1292',
  'Apartment_release_decoration_skeleton_seq135_M1292'],
 34)

In [3]:
sequence_dropdown = widgets.Dropdown(
    options=SEQUENCE_IDS,
    value=SEQUENCE_IDS[0],
    description='sequence',
    layout=widgets.Layout(width='650px'),
)

start_slider = widgets.IntSlider(value=0, min=0, max=100, step=1, description='start')
end_slider = widgets.IntSlider(value=120, min=1, max=120, step=1, description='end')
stride_slider = widgets.IntSlider(value=10, min=1, max=60, step=1, description='stride')
head_scale_slider = widgets.FloatSlider(value=0.35, min=0.05, max=1.5, step=0.05, description='head_scale')
gaze_scale_mode = widgets.Dropdown(options=['fixed', 'depth'], value='fixed', description='gaze_scale')
gaze_fixed_scale_slider = widgets.FloatSlider(value=0.35, min=0.05, max=1.5, step=0.05, description='gaze_len')
show_trajectory_checkbox = widgets.Checkbox(value=True, description='show trajectory')
output = widgets.Output()

def update_bounds(*_):
    frame = get_frame(sequence_dropdown.value)
    max_row = max(len(frame) - 1, 1)
    start_slider.max = max_row
    end_slider.max = len(frame)
    if start_slider.value > max_row:
        start_slider.value = 0
    if end_slider.value <= start_slider.value:
        end_slider.value = min(len(frame), start_slider.value + 120)

def render(*_):
    frame = get_frame(sequence_dropdown.value)
    end_value = max(end_slider.value, start_slider.value + 1)
    window = slice_frame_window(
        frame,
        start_row=start_slider.value,
        end_row=end_value,
        stride=stride_slider.value,
    )
    with output:
        clear_output(wait=True)
        fig = plot_gaze_head_scene_window(
            window,
            head_scale_m=head_scale_slider.value,
            gaze_scale_mode=gaze_scale_mode.value,
            fixed_gaze_scale_m=gaze_fixed_scale_slider.value,
            show_trajectory=show_trajectory_checkbox.value,
        )
        display(fig)
        plt.close(fig)

sequence_dropdown.observe(update_bounds, names='value')
for widget in [
    sequence_dropdown,
    start_slider,
    end_slider,
    stride_slider,
    head_scale_slider,
    gaze_scale_mode,
    gaze_fixed_scale_slider,
    show_trajectory_checkbox,
]:
    widget.observe(render, names='value')

update_bounds()
render()

display(
    widgets.VBox([
        sequence_dropdown,
        widgets.HBox([start_slider, end_slider, stride_slider]),
        widgets.HBox([head_scale_slider, gaze_scale_mode, gaze_fixed_scale_slider]),
        show_trajectory_checkbox,
        output,
    ])
)
